In [1]:
DATA = "../data"
OUTPUT = "../submission"

In [2]:
# Nombres de las acciones

SYMBOLS = {
    "TOT": "Total",
    "XOM": "Exxon",
    "CVX": "Chevron",
    "COP": "ConocoPhillips",
    "VLO": "Valero Energy",
    "MSFT": "Microsoft",
    "IBM": "IBM",
    "TWX": "Time Warner",
    "CMCSA": "Comcast",
    "CVC": "Cablevision",
    "YHOO": "Yahoo",
    "DELL": "Dell",
    "HPQ": "HP",
    "AMZN": "Amazon",
    "TM": "Toyota",
    "CAJ": "Canon",
    "SNE": "Sony",
    "F": "Ford",
    "HMC": "Honda",
    "NAV": "Navistar",
    "NOC": "Northrop Grumman",
    "BA": "Boeing",
    "KO": "Coca Cola",
    "MMM": "3M",
    "MCD": "McDonald's",
    "PEP": "Pepsi",
    "K": "Kellogg",
    "UN": "Unilever",
    "MAR": "Marriott",
    "PG": "Procter Gamble",
    "CL": "Colgate-Palmolive",
    "GE": "General Electrics",
    "WFC": "Wells Fargo",
    "JPM": "JPMorgan Chase",
    "AIG": "AIG",
    "AXP": "American express",
    "BAC": "Bank of America",
    "GS": "Goldman Sachs",
    "AAPL": "Apple",
    "SAP": "SAP",
    "CSCO": "Cisco",
    "TXN": "Texas Instruments",
    "XRX": "Xerox",
    "WMT": "Wal-Mart",
    "HD": "Home Depot",
    "GSK": "GlaxoSmithKline",
    "PFE": "Pfizer",
    "SNY": "Sanofi-Aventis",
    "NVS": "Novartis",
    "KMB": "Kimberly-Clark",
    "R": "Ryder",
    "GD": "General Dynamics",
    "RTN": "Raytheon",
    "CVS": "CVS",
    "CAT": "Caterpillar",
    "DD": "DuPont de Nemours",
}

In [3]:
# Carga de datos:
# Principales acciones entre 2003 y 2008.
#
# Los nombres se leen en el mismo orden que los archivos para que cada
# serie quede asociada a su empresa.


def load_data():

    import numpy as np
    import pandas as pd

    symbols = sorted(SYMBOLS.keys())
    names = np.array([SYMBOLS[symbol] for symbol in symbols])

    quotes = [
        pd.read_csv(f"{DATA}/{symbol}.csv.zip", compression="zip")
        for symbol in symbols
    ]

    return quotes, names


quotes, names = load_data()
len(quotes)

56

In [4]:
# Cálculo de las variaciones diarias (normalizadas):
# se usa la correlación en lugar de la covarianza.


def compute_variations(quotes):

    import numpy as np

    close_prices = np.vstack([q["close"] for q in quotes])
    open_prices = np.vstack([q["open"] for q in quotes])
    variation = close_prices - open_prices

    x = variation.copy().T
    x /= x.std(axis=0)

    return x


x = compute_variations(quotes)
x.shape

(1258, 56)

In [5]:
# Aprendizaje de la estructura gráfica


def make_edge_model(x):

    import numpy as np
    from sklearn.covariance import GraphicalLassoCV

    edge_model = GraphicalLassoCV(alphas=np.logspace(-1.5, 1, num=10))
    edge_model.fit(x)

    return edge_model


edge_model = make_edge_model(x)

In [6]:
# Clustering con Affinity Propagation


def make_clusters(edge_model):

    from sklearn.cluster import affinity_propagation

    _, labels = affinity_propagation(edge_model.covariance_, random_state=0)

    return labels


labels = make_clusters(edge_model)
n_labels = labels.max()

for i in range(n_labels + 1):
    print(f"Cluster {i + 1}: {', '.join(names[labels == i])}")

Cluster 1: Apple, Amazon, Yahoo
Cluster 2: Comcast, Cablevision, Time Warner
Cluster 3: ConocoPhillips, Chevron, Total, Valero Energy, Exxon
Cluster 4: Cisco, Dell, HP, IBM, Microsoft, SAP, Texas Instruments
Cluster 5: Boeing, General Dynamics, Northrop Grumman, Raytheon
Cluster 6: AIG, American express, Bank of America, Caterpillar, CVS, DuPont de Nemours, Ford, General Electrics, Goldman Sachs, Home Depot, JPMorgan Chase, Marriott, McDonald's, 3M, Ryder, Wells Fargo, Wal-Mart
Cluster 7: GlaxoSmithKline, Novartis, Pfizer, Sanofi-Aventis, Unilever
Cluster 8: Kellogg, Coca Cola, Pepsi
Cluster 9: Colgate-Palmolive, Kimberly-Clark, Procter Gamble
Cluster 10: Canon, Honda, Navistar, Sony, Toyota, Xerox


In [7]:
# Construcción del embedding:
# busca la mejor posición de los nodos en un espacio 2D.


def make_embedding(x):

    from sklearn.manifold import LocallyLinearEmbedding

    node_position_model = LocallyLinearEmbedding(
        n_components=2,
        eigen_solver="dense",
        n_neighbors=6,
    )

    return node_position_model.fit_transform(x.T).T


embedding = make_embedding(x)

In [8]:
# Visualización


def plot_stocks(edge_model, embedding, labels, names):

    import os

    import matplotlib.pyplot as plt
    import numpy as np
    from matplotlib.collections import LineCollection

    if not os.path.exists(OUTPUT):
        os.makedirs(OUTPUT)

    n_labels = labels.max()

    plt.figure(1, facecolor="w", figsize=(10, 8))
    plt.clf()
    ax = plt.axes([0.0, 0.0, 1.0, 1.0])
    plt.axis("off")

    # Correlaciones parciales
    partial_correlations = edge_model.precision_.copy()
    d = 1 / np.sqrt(np.diag(partial_correlations))
    partial_correlations *= d
    partial_correlations *= d[:, np.newaxis]
    non_zero = np.abs(np.triu(partial_correlations, k=1)) > 0.02

    # Nodos
    plt.scatter(
        embedding[0],
        embedding[1],
        s=100 * d**2,
        c=labels,
        cmap=plt.cm.nipy_spectral,
    )

    # Bordes
    start_idx, end_idx = np.where(non_zero)
    segments = [
        [embedding[:, start], embedding[:, stop]]
        for start, stop in zip(start_idx, end_idx)
    ]
    values = np.abs(partial_correlations[non_zero])
    lc = LineCollection(
        segments,
        zorder=0,
        cmap=plt.cm.hot_r,
        norm=plt.Normalize(0, 0.7 * values.max()),
    )
    lc.set_array(values)
    lc.set_linewidths(15 * values)
    ax.add_collection(lc)

    # Etiquetas: se ubican para evitar que se traslapen
    for index, (name, label, (x, y)) in enumerate(zip(names, labels, embedding.T)):
        dx = x - embedding[0]
        dx[index] = 1
        dy = y - embedding[1]
        dy[index] = 1
        this_dx = dx[np.argmin(np.abs(dy))]
        this_dy = dy[np.argmin(np.abs(dx))]
        if this_dx > 0:
            horizontalalignment = "left"
            x = x + 0.002
        else:
            horizontalalignment = "right"
            x = x - 0.002
        if this_dy > 0:
            verticalalignment = "bottom"
            y = y + 0.002
        else:
            verticalalignment = "top"
            y = y - 0.002
        plt.text(
            x,
            y,
            name,
            size=10,
            horizontalalignment=horizontalalignment,
            verticalalignment=verticalalignment,
            bbox=dict(
                facecolor="w",
                edgecolor=plt.cm.nipy_spectral(label / float(n_labels)),
                alpha=0.6,
            ),
        )

    plt.xlim(
        embedding[0].min() - 0.15 * np.ptp(embedding[0]),
        embedding[0].max() + 0.10 * np.ptp(embedding[0]),
    )
    plt.ylim(
        embedding[1].min() - 0.03 * np.ptp(embedding[1]),
        embedding[1].max() + 0.03 * np.ptp(embedding[1]),
    )

    plt.savefig(f"{OUTPUT}/stocks.png")
    plt.show()


plot_stocks(edge_model, embedding, labels, names)

<Figure size 1000x800 with 1 Axes>